In [1]:
import os
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
base = "/lakehouse/default/Files"

df = pd.read_excel(f"{base}/seed/Superstore_data_xl.xlsx")
df.columns = [c.strip().lower().replace(" ", "_").replace("-", "_") for c in df.columns]
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
df["ship_date"] = pd.to_datetime(df["ship_date"].astype(str), errors="coerce", format="mixed")

customers = df[["customer_id", "customer_name", "segment", "country", "city",
                "state", "postal_code", "region"]].drop_duplicates("customer_id")

p = df.assign(unit_cost=(df.sales - df.profit) / df.quantity,
              unit_list_price=df.sales / (1 - df.discount) / df.quantity)
products = (p.groupby(["product_id", "category", "sub_category", "product_name"], as_index=False)
             [["unit_cost", "unit_list_price"]].mean().round(2))

orders = df[["order_id", "order_date", "ship_mode", "customer_id", "country",
             "city", "state", "postal_code", "region"]].drop_duplicates("order_id")
order_lines = df[["row_id", "order_id", "product_id", "quantity", "sales", "discount", "profit"]]

# Shipments: ~8% of lines ship short
ship = df[["row_id", "order_id", "product_id", "quantity", "ship_date", "ship_mode"]].copy()
short = rng.random(len(ship)) < 0.08
ship["qty_shipped"] = np.where(short, (ship.quantity * rng.uniform(0.5, 0.9, len(ship))).astype(int), ship.quantity)
ship.insert(0, "shipment_id", "SH-" + ship.row_id.astype(str).str.zfill(6))
ship = ship.rename(columns={"row_id": "order_line_id", "quantity": "qty_ordered"})

# Inventory: weekly snapshot per product x region, ~30 days of demand on hand
weeks = pd.date_range(df.order_date.min().normalize(), df.order_date.max(), freq="W-SUN")
regions = df.region.dropna().unique()
days = max((df.order_date.max() - df.order_date.min()).days, 1)
daily_demand = df.groupby("product_id").quantity.sum() / days
inv = pd.MultiIndex.from_product([weeks, daily_demand.index, regions],
                                 names=["snapshot_date", "product_id", "region"]).to_frame(index=False)
avg = inv.product_id.map(daily_demand) / len(regions)
inv["qty_on_hand"] = np.maximum(0, rng.normal(avg * 30 + 5, 3)).astype(int)
inv["qty_on_order"] = (inv.qty_on_hand * rng.uniform(0, 0.3, len(inv))).astype(int)
inv["snapshot_date"] = inv.snapshot_date.dt.date

def save(pdf, folder, name):
    os.makedirs(f"{base}/{folder}", exist_ok=True)
    pdf.to_csv(f"{base}/{folder}/{name}.csv", index=False)

save(customers, "landing", "customers")
save(products, "landing", "products")
save(inv, "landing", "inventory_snapshots")
save(orders, "seed", "orders")
save(order_lines, "seed", "order_lines")
save(ship, "seed", "shipments")

StatementMeta(, d69bcb46-b68c-40df-a41a-1581f6074462, 3, Finished, Available, Finished, False)